In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the training dataset
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/train.csv')

# Display the first few rows of the dataset
print(train_data.head())

# Get a summary of the dataset
print(train_data.info())

# Check for missing values
print(train_data.isnull().sum())

# Describe the dataset
print(train_data.describe())

# Distinguish column types
numeric_cols = train_data.select_dtypes(include=[np.number]).columns
categorical_cols = train_data.select_dtypes(include=['object', 'category']).columns

print("Numeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

# Visualize the distribution of numeric columns
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(train_data[col], bins=30, kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

# Visualize the distribution of categorical columns
for col in categorical_cols:
    plt.figure(figsize=(8, 4))
    sns.countplot(y=train_data[col])
    plt.title(f'Distribution of {col}')
    plt.xlabel('Count')
    plt.ylabel(col)
    plt.show()

# Correlation matrix for numeric columns
plt.figure(figsize=(12, 8))
corr_matrix = train_data[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()


    id  bone_length  rotting_flesh  hair_length  color    type
0  472     0.681615       0.529227     0.625242  white   Ghoul
1  170     0.480836       0.407930     0.539005  clear  Goblin
2  189     0.375197       0.742953     0.320764   blue   Ghost
3  861     0.626017       0.172182     0.408422   blue   Ghoul
4   30     0.250770       0.246258     0.554654  black   Ghost
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 296 entries, 0 to 295
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             296 non-null    int64  
 1   bone_length    296 non-null    float64
 2   rotting_flesh  296 non-null    float64
 3   hair_length    296 non-null    float64
 4   color          296 non-null    object 
 5   type           296 non-null    object 
dtypes: float64(3), int64(1), object(2)
memory usage: 14.0+ KB
None
id               0
bone_length      0
rotting_flesh    0
hair_length      0
color            0
typ

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


2025-09-15 07:47:13.413 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['color', 'type'], 'Numeric': ['id', 'bone_length', 'rotting_flesh', 'hair_length'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Load the datasets
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/train.csv')
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/test.csv')

# Separate features and target
X_train = train_data.drop(columns=['type'])
y_train = train_data['type']
X_test = test_data.drop(columns=['type'])
y_test = test_data['type']

# Handle missing values
numeric_cols = X_train.select_dtypes(include=[np.number]).columns
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns

# Fill missing values for numeric columns with mean
fill_missing_numeric = FillMissingValue(features=numeric_cols, strategy='mean')
X_train = fill_missing_numeric.fit_transform(X_train)
X_test = fill_missing_numeric.transform(X_test)

# Fill missing values for categorical columns with most frequent
fill_missing_categorical = FillMissingValue(features=categorical_cols, strategy='most_frequent')
X_train = fill_missing_categorical.fit_transform(X_train)
X_test = fill_missing_categorical.transform(X_test)

# Encode categorical variables
label_encode = LabelEncode(features=categorical_cols)
X_train = label_encode.fit_transform(X_train)
X_test = label_encode.transform(X_test)

# Normalize numerical features
standard_scale = StandardScale(features=numeric_cols)
X_train = standard_scale.fit_transform(X_train)
X_test = standard_scale.transform(X_test)

# Display the preprocessed data
print(X_train.head())
print(X_test.head())


         id  bone_length  rotting_flesh  hair_length  color
0  0.070848     1.891869       0.107095     0.579263      6
1 -1.090369     0.363108      -0.696121     0.079077      3
2 -1.017312    -0.441246       1.522363    -1.186733      2
3  1.566589     1.468536      -2.257217    -0.678313      2
4 -1.628682    -1.388649      -1.766698     0.169847      0
         id  bone_length  rotting_flesh  hair_length  color
0  1.251291     0.630880       0.095709    -0.988991      6
1 -1.467188     0.689698      -1.288431    -1.132321      4
2 -1.632527     0.510522      -0.494233     0.041523      3
3  1.120558    -0.120664      -0.896998     0.095487      3
4 -1.286469     0.625331       0.460702     0.251436      3


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(X_train)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'bone_length', 'rotting_flesh', 'hair_length', 'color'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from metagpt.tools.libs.data_preprocess import LabelEncode, StandardScale

# Assuming X_train, X_test, y_train, y_test are already defined from previous tasks

# Initialize and fit the LabelEncoder for categorical columns
label_encode = LabelEncode(features=['color'])
X_train = label_encode.fit_transform(X_train)
X_test = label_encode.transform(X_test)

# Initialize and fit the StandardScaler for numeric columns
standard_scale = StandardScale(features=['id', 'bone_length', 'rotting_flesh', 'hair_length', 'color'])
X_train = standard_scale.fit_transform(X_train)
X_test = standard_scale.transform(X_test)

# Train a RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Print classification report
print(classification_report(y_test, y_pred))

# Save predictions to a CSV file
predictions_df = pd.DataFrame({'id': test_data['id'], 'predicted_type': y_pred})
predictions_df.to_csv('monster_type_predictions.csv', index=False)


Accuracy: 0.60
              precision    recall  f1-score   support

       Ghost       0.72      0.54      0.62        24
       Ghoul       0.71      0.69      0.70        29
      Goblin       0.41      0.55      0.47        22

    accuracy                           0.60        75
   macro avg       0.62      0.59      0.60        75
weighted avg       0.63      0.60      0.61        75



In [6]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

# Load the test data
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/test.csv')

# Preprocess the test data using the same transformations as the training data
X_test = test_data.drop(columns=['type'])
y_test = test_data['type']

# Apply the same preprocessing steps as in the training data
X_test = fill_missing_numeric.transform(X_test)
X_test = fill_missing_categorical.transform(X_test)
X_test = label_encode.transform(X_test)
X_test = standard_scale.transform(X_test)

# Load the trained model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions on the test data
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(classification_report(y_test, y_pred))

# Save the predictions to a CSV file
predictions_df = pd.DataFrame({'id': test_data['id'], 'predicted_type': y_pred})
predictions_df.to_csv('monster_type_predictions.csv', index=False)


Accuracy: 0.37
              precision    recall  f1-score   support

       Ghost       0.00      0.00      0.00        24
       Ghoul       0.38      0.97      0.55        29
      Goblin       0.00      0.00      0.00        22

    accuracy                           0.37        75
   macro avg       0.13      0.32      0.18        75
weighted avg       0.15      0.37      0.21        75



D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this 

In [7]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Load test data
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/test.csv')

# Preprocess test data
X_test = test_data.drop(columns=['type'])
y_test = test_data['type']

# Fill missing values
X_test = fill_missing_numeric.transform(X_test)
X_test = fill_missing_categorical.transform(X_test)

# Label encode categorical columns
X_test = label_encode.transform(X_test)

# Standard scale numeric columns
X_test = standard_scale.transform(X_test)

# Load the trained model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on the test data
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Print classification report
print(classification_report(y_test, y_pred))

# Save predictions to a CSV file
predictions_df = pd.DataFrame({'id': test_data['id'], 'predicted_type': y_pred})
predictions_df.to_csv('monster_type_predictions.csv', index=False)


Accuracy: 0.37
              precision    recall  f1-score   support

       Ghost       0.00      0.00      0.00        24
       Ghoul       0.38      0.97      0.55        29
      Goblin       0.00      0.00      0.00        22

    accuracy                           0.37        75
   macro avg       0.13      0.32      0.18        75
weighted avg       0.15      0.37      0.21        75



D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this 